In [68]:
import pandas as pd
import json

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [ ]:
df = pd.read_csv("data/en_10_gpt_preds_subsentences_phrasal_verbs.csv")

rows = list()
for idx, row in df.iterrows():
    preds = json.loads(row["preds"])
    for token in preds:
        temp = dict()
        temp["sent"] = row["sents"]
        for k in token.keys():
            temp[k] = token[k]
        rows.append(temp)

preds_df = pd.DataFrame(rows).head(4185)

preds_df

In [ ]:
columns = ["ID", "sent", "word", "manual_fee", "manual_frame", "manual_partof", "manual_comment"]
manual_df = pd.read_csv("data/english_fees.csv", header=0, names=columns)

manual_df

In [ ]:
combined_df = pd.concat([preds_df, manual_df[["manual_fee", "manual_frame", "manual_partof", "manual_comment"]]], axis=1)

combined_df

Total non-matches, including fee=False:

In [ ]:
combined_df[combined_df["frame"]!=combined_df["manual_frame"]]

In [73]:
combined_df[~combined_df["manual_comment"].isna()]

,sent,word,fee,frame,partof,manual_fee,manual_frame,manual_partof,manual_comment


In [74]:
print(f"""Complete match at {combined_df[combined_df["frame"]==combined_df["manual_frame"]].shape[0] / combined_df.shape[0] * 100}%""")

Complete match at 85.04181600955795%


Total non-matches, excluding fee=False:

In [ ]:
combined_df[(combined_df["frame"]!=combined_df["manual_frame"])&(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)]

In [76]:
print(f"""Only fee=True complete match at {combined_df[(combined_df["frame"]==combined_df["manual_frame"])&(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)].shape[0] / combined_df[(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)].shape[0] * 100}%""")

Only fee=True complete match at 72.69919703520692%


Percentage of To_Review FEEs:

In [77]:
print(f"""For automatic annotation: {combined_df[combined_df["frame"]=="To_Review"].shape[0] / combined_df[combined_df["fee"]==True].shape[0] * 100}%""")

For automatic annotation: 0.05938242280285036%


In [78]:
print(f"""For manual annotation: {combined_df[combined_df["manual_frame"]=="To_Review"].shape[0] / combined_df[combined_df["manual_fee"]==True].shape[0] * 100}%""")

For manual annotation: 2.372685185185185%


Percentage of manual FEEs:

In [79]:
print(f"""For manual annotation: {combined_df[combined_df["manual_fee"]==True].shape[0] / combined_df.shape[0] * 100}%""")

For manual annotation: 41.29032258064516%


In [80]:
print(f"""Accuracy: {accuracy_score(combined_df["manual_frame"], combined_df["frame"])}""")

Accuracy: 0.8504181600955795


In [81]:
print(f"""Micro metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
""")

Micro metrics:

Precision: 0.8504181600955795
Recall: 0.8504181600955795
F1-score: 0.8504181600955795



In [82]:
print(f"""Macro metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
""")

Macro metrics:

Precision: 0.5564204871279382
Recall: 0.5405328806024321
F1-score: 0.5258338810109902



In [83]:
print(f"""Weighted metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
""")

Weighted metrics:

Precision: 0.8801072101931643
Recall: 0.8504181600955795
F1-score: 0.8458410971335402



In [84]:
combined_df[["manual_frame"]].value_counts()

manual_frame  
-                 2447
People              86
Communication       51
To_Review           41
Law                 39
                  ... 
Likelihood           1
Attack               1
Burying              1
Assistance           1
Historic_event       1
Name: count, Length: 453, dtype: int64